In [0]:
#to download file from the source, we're requesting this module
import urllib.request

#this is the location of the file available as a public GitHub link for NYC TLC Taxi Zone Lookup
url = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/nyc-tlc/taxi%2B_zone_lookup.csv"

#we're basically telling databricks who's the current user
user = spark.sql("SELECT current_user()").collect()[0][0]

#and we dynamically put the user's name in the f-string placeholder
#we're doing this to create a temporary/local copy of the file inside our databricks workspace
path = f"/Workspace/Users/{user}/taxi_zone_lookup.csv"

#now we download the csv file
urllib.request.urlretrieve(url, path)

#we're creating a dataframe sdf and having spark read it with the headers and also their data types and as a csv file.
sdf = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:" + path)

#displays the dataframe
display(sdf)

#displays the structure of the dataframe
sdf.printSchema()

#we're creating a database but dont throw an error if it already exists
spark.sql("CREATE DATABASE IF NOT EXISTS demodb")

#Remove old table if it exists
spark.sql("DROP TABLE IF EXISTS demodb.taxi_zones")

#we're writing the dataframe to a delta table and if it already exists then we ask it to overwrite
sdf.write.mode("overwrite").saveAsTable("demodb.taxi_zones")